In [1]:
#imports
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from typing import Literal

from src.data.dataset import MetrDataset
from src.data.window_dataset import WindowDataset
from src.model.basic_model.static import StaticModel
from src.model.basic_model.ha import HAStaticModel
from src.data.analysis import get_historical_average, make_df
from src.utils.metrics.metrics import masked_mae, masked_mse


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
def evaluate_model(model: nn.Module, device: torch.device, split: Literal["train", "val", "test"], horizon: int):
    res = []
    ds = WindowDataset(MetrDataset("data/METR-LA", split=split), out_size=12)
    loader = DataLoader(ds, batch_size=len(ds), shuffle=False)
    mae = mse = 0.0
    with torch.no_grad():
        for _, X, output_ts, y in loader:
            X, y, output_ts = X.to(device), y.to(device), output_ts.to(device)
            pred = model(X, output_ts, horizon)
            mae += masked_mae(pred[:, horizon - 1], y[:, horizon - 1], 0.0).item()
            mse += masked_mse(pred[:, horizon - 1], y[:, horizon - 1], 0.0).item()
    n = len(loader)
    res.append(
            {
                "minutes": horizon * 5,
                "split": split,
                "MAE": mae / n,
                "RMSE": np.sqrt(mse / n),
            }
        )
    return res

In [4]:
split = "test"

In [5]:
import importlib, src.model.basic_model.static as static
importlib.reload(static)
from src.model.basic_model.static import StaticModel

static_model = StaticModel().to(device)
rows = [r for h in [1, 3, 6, 12] for r in evaluate_model(static_model, device, "test", h)]
static_model_df = pd.DataFrame(rows).set_index(["split", "minutes"]).round(3)
static_model_df

c:\polibooks\an4\thesis\src\data\dataset.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.sensor_ids, self.id_to_ind, self.adj_mx = pickle.load(f, encoding="latin1")


MAE    RMSE
split minutes               
test  5        2.887   5.136
      15       3.715   7.471
      30       4.619   9.635
      60       6.143  12.611

In [9]:
static_model_2 = StaticModel(last_valid=False).to(device)
rows = [r for h in [1, 3, 6, 12] for r in evaluate_model(static_model_2, device, "test", h)]
static_model_2_df = pd.DataFrame(rows).set_index(["split", "minutes"]).round(3)
static_model_2_df


c:\polibooks\an4\thesis\src\data\dataset.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.sensor_ids, self.id_to_ind, self.adj_mx = pickle.load(f, encoding="latin1")


MAE    RMSE
split minutes               
test  5        3.050   6.081
      15       4.021   8.703
      30       5.100  11.145
      60       6.803  14.223

In [10]:
dataset = MetrDataset("data/METR-LA", split="train")
df = make_df(dataset.get_data(), dataset.get_timestamps())
ha_model = HAStaticModel(torch.tensor(get_historical_average(df), dtype=torch.float32).to(device)).to(device)
rows = [r for h in [1, 3, 6, 12] for r in evaluate_model(ha_model, device, "test", h)]
ha_model_df = pd.DataFrame(rows).set_index(["split", "minutes"]).round(3)
ha_model_df

c:\polibooks\an4\thesis\src\data\dataset.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  self.sensor_ids, self.id_to_ind, self.adj_mx = pickle.load(f, encoding="latin1")


MAE   RMSE
split minutes              
test  5        4.193  7.862
      15       4.193  7.862
      30       4.193  7.862
      60       4.193  7.862

In [11]:
results = pd.concat([static_model_2_df, static_model_df, ha_model_df], axis=1, keys=["static_simple", "static_valid", "ha"])
results = results.droplevel("split")
results

static_simple         static_valid             ha       
                  MAE    RMSE          MAE    RMSE    MAE   RMSE
minutes                                                         
5               3.050   6.081        2.887   5.136  4.193  7.862
15              4.021   8.703        3.715   7.471  4.193  7.862
30              5.100  11.145        4.619   9.635  4.193  7.862
60              6.803  14.223        6.143  12.611  4.193  7.862